In [20]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier


import spacy



In [2]:

columns = ['id','country','Label','Text']
df = pd.read_csv("twitter_training.csv", names=columns)
df.head()

,id,country,Label,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       74682 non-null  int64 
 1   country  74682 non-null  object
 2   Label    74682 non-null  object
 3   Text     73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [4]:
df.head()

,id,country,Label,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [5]:
nlp = spacy.load("en_core_web_sm") 


In [6]:
def preprocess(text):
    # remove stop words and lemmatize the text
    doc = nlp(text)
    filtered_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
    
    return " ".join(filtered_tokens) 

In [11]:
df['Text'] = df['Text'].astype(str).fillna("")
df['Text']

0        im getting on borderlands and i will murder yo...
1        I am coming to the borders and I will kill you...
2        im getting on borderlands and i will kill you ...
3        im coming on borderlands and i will murder you...
4        im getting on borderlands 2 and i will murder ...
                               ...                        
74677    Just realized that the Windows partition of my...
74678    Just realized that my Mac window partition is ...
74679    Just realized the windows partition of my Mac ...
74680    Just realized between the windows partition of...
74681    Just like the windows partition of my Mac is l...
Name: Text, Length: 74682, dtype: object

In [12]:
df['Preprocessed Text'] = df['Text'].apply(preprocess)


In [13]:
df

,id,country,Label,Text,Preprocessed Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,m get borderland murder
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,come border kill
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,m get borderland kill
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,m come borderland murder
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,m get borderland 2 murder
...,...,...,...,...,...
74677,9200,Nvidia,Positive,Just realized that the Windows partition of my...,realize Windows partition Mac like 6 year Nvid...
74678,9200,Nvidia,Positive,Just realized that my Mac window partition is ...,realize Mac window partition 6 year Nvidia dri...
74679,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...,realize window partition Mac 6 year Nvidia dri...
74680,9200,Nvidia,Positive,Just realized between the windows partition of...,realize window partition Mac like 6 year Nvidi...


In [14]:
le_model = LabelEncoder()
df['Label'] = le_model.fit_transform(df['Label'])

In [16]:
df.head()


,id,country,Label,Text,Preprocessed Text
0,2401,Borderlands,3,im getting on borderlands and i will murder yo...,m get borderland murder
1,2401,Borderlands,3,I am coming to the borders and I will kill you...,come border kill
2,2401,Borderlands,3,im getting on borderlands and i will kill you ...,m get borderland kill
3,2401,Borderlands,3,im coming on borderlands and i will murder you...,m come borderland murder
4,2401,Borderlands,3,im getting on borderlands 2 and i will murder ...,m get borderland 2 murder


In [17]:
X_train, X_test, y_train, y_test = train_test_split(df['Preprocessed Text'], df['Label'], 
                                                    test_size=0.2, random_state=42, stratify=df['Label'])

In [21]:
v = TfidfVectorizer()

X_train_cv = v.fit_transform(X_train)
X_test_cv = v.transform(X_test)

In [25]:
model = KNeighborsClassifier(n_neighbors = 4)

model.fit(X_train_cv, y_train)

y_pred = model.predict(X_test_cv)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.87      0.90      2598
           1       0.81      0.96      0.88      4509
           2       0.95      0.90      0.92      3664
           3       0.95      0.86      0.90      4166

    accuracy                           0.90     14937
   macro avg       0.91      0.90      0.90     14937
weighted avg       0.91      0.90      0.90     14937



In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_cv, y_train)
y_pred = model.predict(X_test_cv)
print("Logistic Regression:\n", classification_report(y_test, y_pred))


Logistic Regression:
               precision    recall  f1-score   support

           0       0.83      0.68      0.75      2598
           1       0.80      0.83      0.82      4509
           2       0.79      0.75      0.77      3664
           3       0.74      0.83      0.78      4166

    accuracy                           0.79     14937
   macro avg       0.79      0.77      0.78     14937
weighted avg       0.79      0.79      0.78     14937



In [27]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_cv, y_train)
y_pred = model.predict(X_test_cv)
print("Decision Tree:\n", classification_report(y_test, y_pred))


Decision Tree:
               precision    recall  f1-score   support

           0       0.81      0.73      0.77      2598
           1       0.85      0.83      0.84      4509
           2       0.82      0.79      0.80      3664
           3       0.76      0.85      0.80      4166

    accuracy                           0.81     14937
   macro avg       0.81      0.80      0.80     14937
weighted avg       0.81      0.81      0.81     14937



In [28]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_cv, y_train)
y_pred = model.predict(X_test_cv)
print("Random Forest:\n", classification_report(y_test, y_pred))


Random Forest:
               precision    recall  f1-score   support

           0       0.97      0.84      0.90      2598
           1       0.93      0.92      0.93      4509
           2       0.92      0.90      0.91      3664
           3       0.85      0.95      0.90      4166

    accuracy                           0.91     14937
   macro avg       0.92      0.90      0.91     14937
weighted avg       0.91      0.91      0.91     14937



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_cv.shape[1],)), 
    Dropout(0.3),  
    Dense(64, activation='relu'),  
    Dropout(0.3),
    Dense(len(set(y_train)), activation='softmax') 
])





In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:

history = model.fit(X_train_cv, y_train, 
                    validation_data=(X_test_cv, y_test), 
                    epochs=20, batch_size=32, verbose=1)

y_pred = model.predict(X_test_cv)
y_pred_classes = y_pred.argmax(axis=1)


In [ ]:

print("ANN Classification Report:\n", classification_report(y_test, y_pred_classes))
